# Error Analysis and System Insights

This notebook works at the prediction-row level using `outputs/benchmark_predictions.csv`. It is analysis-only and does not call APIs or models.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from either repo-root or notebooks/ execution."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "SMSSpamCollection").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find project root containing data/SMSSpamCollection and src/.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "SMSSpamCollection"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
print(f"Project root: {PROJECT_ROOT.relative_to(PROJECT_ROOT)} (resolved internally)")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

PREDICTIONS_PATH = OUTPUTS_DIR / "benchmark_predictions.csv"
HAVE_FILES = PREDICTIONS_PATH.exists()
if not HAVE_FILES:
    display(Markdown("""**Missing prediction artifact.** Run this from the repo root before using this notebook:

```bash
python -m src.benchmark --modes tfidf_lr,base_llm,basic_rag,advanced_base,advanced_lora,guarded_fallback --sample-size 75
```"""))
else:
    predictions = pd.read_csv(PREDICTIONS_PATH)
    print("Prediction artifact found.")

## Prediction Rows and Modes

The benchmark writes one row per `(mode, sms)` pair, enabling same-message comparisons across systems.

In [ ]:
if HAVE_FILES:
    print(f"Rows: {len(predictions):,}")
    modes = predictions["mode"].drop_duplicates().tolist()
    print(f"Modes: {modes}")
    display(predictions.head())

## Errors by Mode

Runtime errors are captured as `unknown` predictions with an error message. This benchmark run should have no model/runtime errors.

In [ ]:
if HAVE_FILES:
    error_summary = (
        predictions.assign(has_error=predictions["error_message"].fillna("").astype(str).str.len() > 0)
        .groupby("mode")["has_error"]
        .agg(error_count="sum", rows="count")
        .reset_index()
    )
    display(error_summary)

## False Positives and False Negatives

False negatives are especially important: they represent true spam that the system labeled as ham or unknown.

In [ ]:
if HAVE_FILES:
    errors = predictions[predictions["true_label"] != predictions["predicted_label"]].copy()
    fp = predictions[(predictions["true_label"] == "ham") & (predictions["predicted_label"] == "spam")].copy()
    fn = predictions[(predictions["true_label"] == "spam") & (predictions["predicted_label"] != "spam")].copy()

    error_counts = (
        predictions.assign(
            false_positive=(predictions["true_label"] == "ham") & (predictions["predicted_label"] == "spam"),
            false_negative=(predictions["true_label"] == "spam") & (predictions["predicted_label"] != "spam"),
        )
        .groupby("mode")[["false_positive", "false_negative"]]
        .sum()
        .astype(int)
        .reset_index()
    )
    display(error_counts)

    ax = error_counts.plot(kind="bar", x="mode", y=["false_positive", "false_negative"], color=["#F58518", "#E45756"])
    ax.set_title("Error Types by Mode")
    ax.set_xlabel("Mode")
    ax.set_ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

Advanced LoRA's main issue is false negatives: spam messages are being labeled as ham. The grounded RAG modes and guarded fallback avoid those misses in this sample.

## Advanced LoRA vs Guarded Fallback

Compare the two modes on the same `sms_id` and `original_index` to identify LoRA failures recovered by the fallback path.

In [ ]:
if HAVE_FILES:
    lora = predictions[predictions["mode"] == "advanced_lora"].copy()
    guarded = predictions[predictions["mode"] == "guarded_fallback"].copy()
    compare_cols = ["sms_id", "original_index", "text", "true_label", "predicted_label"]
    comparison = lora[compare_cols].merge(
        guarded[["sms_id", "original_index", "predicted_label", "fallback_used", "initial_lora_prediction", "lora_verification"]],
        on=["sms_id", "original_index"],
        suffixes=("_lora", "_guarded"),
    )
    comparison["guarded_correct"] = comparison["predicted_label_guarded"] == comparison["true_label"]
    display(comparison.head(10))

## LoRA Spam Misses Corrected by Guarded Fallback

These are the most important system-level wins: LoRA predicts ham for true spam, then the guarded fallback returns spam.

In [ ]:
if HAVE_FILES:
    corrected = comparison[
        (comparison["true_label"] == "spam")
        & (comparison["predicted_label_lora"] == "ham")
        & (comparison["predicted_label_guarded"] == "spam")
    ].copy()
    print(f"Corrected LoRA spam misses: {len(corrected)}")
    display(corrected[[
        "sms_id",
        "original_index",
        "true_label",
        "predicted_label_lora",
        "predicted_label_guarded",
        "fallback_used",
        "lora_verification",
        "text",
    ]])

## 75-Sample Rescue Summary

The official 75-sample benchmark contains 10 spam messages. In this run, Advanced LoRA misses 9 spam messages and Guarded Fallback rescues all 9 of those misses. Fallback is used on 11 total rows: 9 spam and 2 ham.

In [ ]:
if HAVE_FILES:
    tfidf_rows = predictions[predictions["mode"] == "tfidf_lr"]
    spam_count = int((tfidf_rows["true_label"] == "spam").sum())

    lora_summary = predictions[predictions["mode"] == "advanced_lora"][["sms_id", "original_index", "text", "true_label", "predicted_label"]].copy()
    guarded_summary = predictions[predictions["mode"] == "guarded_fallback"][["sms_id", "original_index", "predicted_label", "fallback_used", "initial_lora_prediction", "lora_verification"]].copy()
    merged_summary = lora_summary.merge(
        guarded_summary,
        on=["sms_id", "original_index"],
        suffixes=("_lora", "_guarded"),
    )
    spam_misses_summary = merged_summary[
        (merged_summary["true_label"] == "spam")
        & (merged_summary["predicted_label_lora"] != "spam")
    ]
    rescued_summary = spam_misses_summary[spam_misses_summary["predicted_label_guarded"] == "spam"]

    guarded_rows = predictions[predictions["mode"] == "guarded_fallback"].copy()
    guarded_rows["fallback_used"] = guarded_rows["fallback_used"].fillna(False).astype(bool)
    fallback_rows = guarded_rows[guarded_rows["fallback_used"]]

    print(f"Spam examples: {spam_count}")
    print(f"LoRA spam misses: {len(spam_misses_summary)}")
    print(f"Guarded fallback rescued: {len(rescued_summary)}")
    print(f"Fallback-used rows: {len(fallback_rows)}")
    display(fallback_rows["true_label"].value_counts().rename_axis("true_label").reset_index(name="count"))

Evidence-aware prompting alone did not fix the LoRA ham bias in this benchmark. The verifier plus base-agent fallback is what recovered the missed spam cases.

## Guarded Fallback Usage

Inspect rows where fallback was used, then check whether final predictions were correct.

In [ ]:
if HAVE_FILES:
    guarded_rows = predictions[predictions["mode"] == "guarded_fallback"].copy()
    guarded_rows["fallback_used"] = guarded_rows["fallback_used"].fillna(False).astype(bool)
    fallback_rows = guarded_rows[guarded_rows["fallback_used"]].copy()
    fallback_rows["correct"] = fallback_rows["true_label"] == fallback_rows["predicted_label"]

    print(f"Fallback-used rows: {len(fallback_rows)}")
    if len(fallback_rows):
        display(fallback_rows["true_label"].value_counts().rename_axis("true_label").reset_index(name="count"))
        display(fallback_rows["correct"].value_counts().rename_axis("correct").reset_index(name="count"))
        display(fallback_rows[["sms_id", "original_index", "true_label", "predicted_label", "initial_lora_prediction", "lora_verification", "correct", "text"]])
    else:
        display(Markdown("No fallback rows were used in this benchmark output."))

Fallback usage gives a direct measure of how often the verifier distrusted the LoRA path. In this run, fallback rows are where the reliability gain appears.

## Risk Features and Error Patterns

Use logged risk features to summarize what kinds of messages were misclassified.

In [ ]:
if HAVE_FILES:
    risk_cols = [
        "has_url",
        "has_phone",
        "has_currency",
        "has_urgent_word",
        "has_prize_word",
        "has_call_instr",
        "uppercase_ratio",
        "exclamation_count",
        "message_length",
    ]
    error_risk = errors.groupby("mode")[risk_cols].mean().round(4)
    if len(error_risk):
        display(error_risk)
    else:
        display(Markdown("No misclassified rows were found."))

    important_flags = ["has_prize_word", "has_currency", "has_urgent_word", "has_call_instr"]
    if len(errors):
        flag_summary = errors.groupby("mode")[important_flags].sum().astype(int).reset_index()
        display(flag_summary)
    else:
        display(Markdown("No error flag summary to display."))

Misclassified spam should be inspected for prize, currency, urgency, and call/reply cues. In this benchmark, those cues help explain why the LoRA misses are concerning and why fallback correction matters.

## System Insights

- LoRA is locally integrated but biased toward ham on this sample.
- Evidence-aware prompting alone did not fix LoRA.
- Verifier plus fallback provides the actual reliability improvement.
- RAG grounding improves over the base LLM.
- The classical TF-IDF baseline remains competitive and fast, making it a strong benchmark anchor.